# Adversarial RF Robustness — Google Colab Runner

Upload the entire `adversarial-rf-robustness/` folder to Google Drive, then run this notebook with a GPU runtime.

**Runtime > Change runtime type > T4 GPU (or better)**

In [1]:
# 1. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# UPDATE THIS PATH to where you uploaded the project
PROJECT_DIR = '/content/drive/MyDrive/ew project/adversarial-rf-robustness'

import os
os.chdir(PROJECT_DIR)
print(f'Working directory: {os.getcwd()}')
print(f'Files: {os.listdir(".")}')

MessageError: Error: credential propagation was unsuccessful

In [ ]:
# 2. Install dependencies
!pip install scipy pandas matplotlib h5py pyyaml tqdm scikit-learn

In [ ]:
# 3. Verify GPU
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    #print(f'Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB')

In [ ]:
# 4. Run smoke test
!python smoke_test.py --data_path data/raw/RML2016.10a_dict.pkl

In [ ]:
# 5. Phase 0: Baseline CNN Training (~5 min on T4)
!python train.py \
  --data_path data/raw/RML2016.10a_dict.pkl \
  --dataset_version 2016.10a \
  --epochs 60 \
  --batch_size 256 \
  --lr 0.001 \
  --seed 42 \
  --save_dir experiments/results/baseline_cnn \
  --device auto

In [ ]:
# 6. Phase 1: Clean Accuracy vs SNR
!python evaluate.py \
  --model_path experiments/results/baseline_cnn/best_model.pth \
  --data_path data/raw/RML2016.10a_dict.pkl \
  --eval_mode clean_snr \
  --num_trials 10 \
  --output_dir experiments/results/baseline_cnn \
  --device auto \
  --seed 42

In [ ]:
# 7. Phase 2: Attack evaluations (FGSM + PGD across channels)
for attack in ['fgsm', 'pgd']:
    for channel in ['awgn', 'rayleigh_awgn', 'rayleigh_cfo_awgn']:
        print(f'\n--- {attack.upper()} / {channel} ---')
        !python evaluate.py \
          --model_path experiments/results/baseline_cnn/best_model.pth \
          --data_path data/raw/RML2016.10a_dict.pkl \
          --eval_mode attack_snr \
          --attack_type {attack} \
          --channel_type {channel} \
          --rho 0.01 \
          --pgd_steps 10 \
          --num_trials 10 \
          --output_dir experiments/results/baseline_cnn \
          --device auto \
          --seed 42

In [ ]:
# 8. Phase 2: Attack vs Perturbation Budget
for attack in ['fgsm', 'pgd']:
    print(f'\n--- Budget sweep: {attack.upper()} ---')
    !python evaluate.py \
      --model_path experiments/results/baseline_cnn/best_model.pth \
      --data_path data/raw/RML2016.10a_dict.pkl \
      --eval_mode attack_budget \
      --attack_type {attack} \
      --channel_type awgn \
      --pgd_steps 10 \
      --num_trials 10 \
      --output_dir experiments/results/baseline_cnn \
      --device auto \
      --seed 42

In [ ]:
# 9. Phase 3: Defense Training & Evaluation (~20 min on T4)
!python train_defenses.py \
  --data_path data/raw/RML2016.10a_dict.pkl \
  --epochs 60 \
  --batch_size 256 \
  --lr 0.001 \
  --seed 42 \
  --rho 0.01 \
  --pgd_steps 5 \
  --sigma 0.01 \
  --output_dir experiments/results \
  --eval_snr 0 10 \
  --device auto

In [ ]:
# 10. Phase 4: Per-Class Vulnerability Analysis
!python eval_per_class.py \
  --model_path experiments/results/baseline_cnn/best_model.pth \
  --data_path data/raw/RML2016.10a_dict.pkl \
  --snr 0 10 18 \
  --rho 0.01 \
  --attack pgd \
  --pgd_steps 10 \
  --num_trials 10 \
  --output_dir experiments/results \
  --device auto

In [ ]:
# 11. Generate Figures
!python plot_results.py \
  --results_dir experiments/results \
  --output_dir experiments/figures

In [ ]:
# 12. View results
import pandas as pd

print('=== Defense Comparison ===')
df = pd.read_csv('experiments/results/defense_comparison.csv')
print(df.to_string(index=False))

print('\n=== Per-Class Vulnerability (SNR=10) ===')
df2 = pd.read_csv('experiments/results/per_class_vulnerability.csv')
df2_10 = df2[df2['snr'] == 10].sort_values('asr', ascending=False)
print(df2_10[['modulation', 'clean_acc', 'robust_acc', 'asr']].to_string(index=False))

In [ ]:
# 13. Display figures
from IPython.display import Image, display
import glob

for fig_path in sorted(glob.glob('experiments/figures/*.png')):
    print(f'\n{os.path.basename(fig_path)}')
    display(Image(filename=fig_path, width=600))

---

# 2018.01a Dataset Experiments

The sections below replicate the full pipeline on the **RadioML 2018.01a** dataset (24 modulation classes, 1024 I/Q samples per example).

**Data:** Download `2018.01` from [DeepSig](https://www.deepsig.ai/datasets) and place `GOLD_XYZ_OSC.0001_1024.hdf5` in `data/raw/archive-3/2018.01/`.

In [ ]:
# 14. 2018 Baseline CNN Training
DATA_2018 = 'data/raw/archive-3/2018.01/GOLD_XYZ_OSC.0001_1024.hdf5'

!python train.py \
  --data_path {DATA_2018} \
  --dataset_version 2018.01a \
  --num_classes 24 \
  --input_length 1024 \
  --epochs 60 \
  --batch_size 256 \
  --lr 0.001 \
  --seed 42 \
  --save_dir experiments/results_2018/baseline_cnn \
  --device auto

In [ ]:
# 15. 2018 Clean Accuracy vs SNR
!python evaluate.py \
  --model_path experiments/results_2018/baseline_cnn/best_model.pth \
  --data_path {DATA_2018} \
  --dataset_version 2018.01a \
  --num_classes 24 \
  --input_length 1024 \
  --eval_mode clean_snr \
  --num_trials 10 \
  --output_dir experiments/results_2018/baseline_cnn \
  --device auto \
  --seed 42

In [ ]:
# 16. 2018 Attack Evaluations (FGSM + PGD across channels)
for attack in ['fgsm', 'pgd']:
    for channel in ['awgn', 'rayleigh_awgn', 'rayleigh_cfo_awgn']:
        print(f'\n--- {attack.upper()} / {channel} ---')
        !python evaluate.py \
          --model_path experiments/results_2018/baseline_cnn/best_model.pth \
          --data_path {DATA_2018} \
          --dataset_version 2018.01a \
          --num_classes 24 \
          --input_length 1024 \
          --eval_mode attack_snr \
          --attack_type {attack} \
          --channel_type {channel} \
          --rho 0.01 \
          --pgd_steps 10 \
          --num_trials 10 \
          --output_dir experiments/results_2018/baseline_cnn \
          --device auto \
          --seed 42

In [ ]:
# 17. 2018 Defense Training & Evaluation
!python train_defenses.py \
  --data_path {DATA_2018} \
  --dataset_version 2018.01a \
  --num_classes 24 \
  --input_length 1024 \
  --epochs 60 \
  --batch_size 256 \
  --lr 0.001 \
  --seed 42 \
  --rho 0.01 \
  --pgd_steps 5 \
  --sigma 0.01 \
  --output_dir experiments/results_2018 \
  --eval_snr 0 10 \
  --device auto

In [ ]:
# 18. 2018 Per-Class Vulnerability Analysis
!python eval_per_class.py \
  --model_path experiments/results_2018/baseline_cnn/best_model.pth \
  --data_path {DATA_2018} \
  --dataset_version 2018.01a \
  --num_classes 24 \
  --input_length 1024 \
  --snr 0 10 18 \
  --rho 0.01 \
  --attack pgd \
  --pgd_steps 10 \
  --num_trials 10 \
  --output_dir experiments/results_2018 \
  --device auto

---

# Transferability Analysis

Evaluate whether adversarial examples crafted against the baseline model transfer to defended models. Runs on both the 2016 and 2018 datasets.

In [ ]:
# 19. Transferability Analysis (2016.10a)
!python eval_transferability.py \
  --source_path experiments/results/baseline_cnn/best_model.pth \
  --source_name baseline \
  --target_path \
    experiments/results/channel_aug/best_model.pth \
    experiments/results/adv_train/best_model.pth \
    experiments/results/noise_inject/best_model.pth \
  --target_names "Channel Aug." "Adv. Training" "Noise Inj." \
  --data_path data/raw/RML2016.10a_dict.pkl \
  --dataset_version 2016.10a \
  --snr 0 10 \
  --rho 0.01 \
  --attack pgd \
  --pgd_steps 10 \
  --num_trials 10 \
  --output_dir experiments/results \
  --device auto

In [ ]:
# 20. Transferability Analysis (2018.01a)
!python eval_transferability.py \
  --source_path experiments/results_2018/baseline_cnn/best_model.pth \
  --source_name baseline \
  --target_path \
    experiments/results_2018/channel_aug/best_model.pth \
    experiments/results_2018/adv_train/best_model.pth \
    experiments/results_2018/noise_inject/best_model.pth \
    experiments/results_2018/adv_train_channel/best_model.pth \
  --target_names "Channel Aug." "Adv. Training" "Noise Inj." "Adv.+Chan. Aug." \
  --data_path {DATA_2018} \
  --dataset_version 2018.01a \
  --num_classes 24 \
  --input_length 1024 \
  --snr 0 10 \
  --rho 0.01 \
  --attack pgd \
  --pgd_steps 10 \
  --num_trials 10 \
  --output_dir experiments/results_2018 \
  --device auto

---

# Channel-Aware Attack Comparison (freeze_channel)

Compare PGD attacks with and without channel state information (CSI):
- **No CSI** (default): Each PGD step sees an independent channel realization (stochastic attack)
- **Perfect CSI** (`freeze_channel=True`): All PGD steps use the same fixed channel realization

In [ ]:
# 21. Freeze-Channel Attack Comparison
# Runs PGD with freeze_channel=True on fading channels to compare CSI vs no-CSI attacks
for channel in ['rayleigh_awgn', 'rayleigh_cfo_awgn']:
    print(f'\n--- PGD freeze_channel / {channel} (2016) ---')
    !python evaluate.py \
      --model_path experiments/results/baseline_cnn/best_model.pth \
      --data_path data/raw/RML2016.10a_dict.pkl \
      --eval_mode attack_snr \
      --attack_type pgd \
      --channel_type {channel} \
      --rho 0.01 \
      --pgd_steps 10 \
      --freeze-channel \
      --num_trials 10 \
      --output_dir experiments/results/baseline_cnn \
      --device auto \
      --seed 42

for channel in ['rayleigh_awgn', 'rayleigh_cfo_awgn']:
    print(f'\n--- PGD freeze_channel / {channel} (2018) ---')
    !python evaluate.py \
      --model_path experiments/results_2018/baseline_cnn/best_model.pth \
      --data_path {DATA_2018} \
      --dataset_version 2018.01a \
      --num_classes 24 \
      --input_length 1024 \
      --eval_mode attack_snr \
      --attack_type pgd \
      --channel_type {channel} \
      --rho 0.01 \
      --pgd_steps 10 \
      --freeze-channel \
      --num_trials 10 \
      --output_dir experiments/results_2018/baseline_cnn \
      --device auto \
      --seed 42

---

# Per-Class Precision, Recall & F1

Compute per-class precision, recall, and F1-score for both clean and adversarial predictions.

In [ ]:
# 22. Per-Class Precision/Recall/F1 (2016)
!python eval_per_class.py \
  --model_path experiments/results/baseline_cnn/best_model.pth \
  --data_path data/raw/RML2016.10a_dict.pkl \
  --snr 10 \
  --rho 0.01 \
  --attack pgd \
  --pgd_steps 10 \
  --output_dir experiments/results \
  --device auto

# View results
import pandas as pd
try:
    df_prf = pd.read_csv('experiments/results/per_class_prf_clean.csv')
    print('\n=== Per-Class PRF (Clean, SNR=10) ===')
    print(df_prf.to_string(index=False))

    df_adv = pd.read_csv('experiments/results/per_class_prf_adv.csv')
    print('\n=== Per-Class PRF (Adversarial, SNR=10) ===')
    print(df_adv.to_string(index=False))
except FileNotFoundError:
    print('PRF CSV not found — run cell 10 first.')

In [ ]:
# 23. View 2018 Results
import pandas as pd
import glob

print('=== 2018 Defense Comparison ===')
try:
    df = pd.read_csv('experiments/results_2018/defense_comparison.csv')
    print(df.to_string(index=False))
except FileNotFoundError:
    print('Not found — run cell 17 first.')

print('\n=== 2018 Transferability ===')
try:
    df_t = pd.read_csv('experiments/results_2018/transferability_analysis.csv')
    print(df_t.to_string(index=False))
except FileNotFoundError:
    print('Not found — run cell 20 first.')

print('\n=== 2016 Transferability ===')
try:
    df_t16 = pd.read_csv('experiments/results/transferability_analysis.csv')
    print(df_t16.to_string(index=False))
except FileNotFoundError:
    print('Not found — run cell 19 first.')